# 面试题：如何从零实现 MMoE，并诊断多任务推荐中的负迁移？

## 面试回答主线

MMoE 用多个共享 expert 产生不同表示，每个任务通过自己的 gate 对 expert 输出加权，再接任务 tower。它允许点击任务偏向“兴趣”专家、转化任务偏向“价格匹配”专家，可能缓解共享表示中的任务干扰；是否真的改善必须与参数量接近的 shared-bottom 比较，并检查任务梯度夹角。核心计算是 `expert_outputs[B,E,H]`、`gate[B,E]` 和按 expert 维加权求和；gate 的 softmax 必须沿 expert 维，而不是 batch 维。训练还要处理缺失标签、任务 loss 权重、样本选择偏差与 expert collapse。下面不用推荐框架或现成 MMoE 层，只用参数矩阵手写模型、forward、反向传播和门控诊断。

## 真实案例：商品曝光的点击与转化联合预估

输入包含“用户兴趣匹配分”“价格匹配分”“移动端上下文”。点击标签主要由兴趣正负决定，转化标签主要由价格匹配正负决定。训练集覆盖 32 条组合曝光，测试集包含 8 条新强度组合；字段和任务关系模拟真实推荐漏斗，但标签是受控生成，不能替代线上日志或因果评估。

In [1]:
import math  # 导入平方根用于手写参数初始化。
import torch  # 导入 PyTorch 以构造多任务数据并真实训练。
from torch import nn  # 导入基础模块与参数容器。
import torch.nn.functional as F  # 导入底层二元交叉熵。
torch.set_num_threads(1)  # 固定小型多任务实验为单线程。
torch.manual_seed(97)  # 固定参数初始化与数据顺序。
train_records = []  # 收集三字段训练曝光。
for interest in (-1.2, -0.6, 0.6, 1.2):  # 遍历四档兴趣匹配强度。
    for price_fit in (-1.2, -0.6, 0.6, 1.2):  # 遍历四档价格匹配强度。
        for mobile in (-0.25, 0.25):  # 为每个组合加入两种设备上下文。
            click = int(interest > 0.0)  # 用兴趣方向生成点击教学标签。
            conversion = int(price_fit > 0.0)  # 用价格匹配方向生成转化教学标签。
            train_records.append((interest, price_fit, mobile, click, conversion))  # 保存完整训练曝光。
test_records = [  # 定义八条新强度组合的留出曝光。
    (-0.9, -0.9, 0.1, 0, 0),  # 低兴趣低价格匹配。
    (-0.9, 0.9, -0.1, 0, 1),  # 不点击倾向但价格匹配。
    (0.9, -0.9, 0.1, 1, 0),  # 有兴趣但价格不匹配。
    (0.9, 0.9, -0.1, 1, 1),  # 两个任务都为正。
    (-0.35, 1.35, 0.2, 0, 1),  # 弱负兴趣强正价格。
    (0.35, -1.35, -0.2, 1, 0),  # 弱正兴趣强负价格。
    (1.35, 0.35, 0.2, 1, 1),  # 强正兴趣弱正价格。
    (-1.35, -0.35, -0.2, 0, 0),  # 强负兴趣弱负价格。
]  # 完成八条留出测试曝光。
def tensors_from_records(records):  # 把业务记录转换成模型输入与双任务标签。
    features = torch.tensor([[row[0], row[1], row[2]] for row in records], dtype=torch.float32)  # 创建兴趣、价格、设备三维输入。
    targets = torch.tensor([[row[3], row[4]] for row in records], dtype=torch.float32)  # 创建点击与转化双标签。
    return features, targets  # 返回共享输入和两个任务监督。
train_features, train_targets = tensors_from_records(train_records)  # 张量化三十二条训练曝光。
test_features, test_targets = tensors_from_records(test_records)  # 张量化八条留出曝光。
print("编号  兴趣匹配  价格匹配  移动端  点击  转化")  # 输出可读输入预览表头。
for index, record in enumerate(test_records, start=1):  # 遍历全部测试曝光。
    print(f"{index:>2}     {record[0]:>5.2f}      {record[1]:>5.2f}     {record[2]:>5.2f}    {record[3]}     {record[4]}")  # 展示两个任务可能一致或冲突的样本。

编号  兴趣匹配  价格匹配  移动端  点击  转化
 1     -0.90      -0.90      0.10    0     0
 2     -0.90       0.90     -0.10    0     1
 3      0.90      -0.90      0.10    1     0
 4      0.90       0.90     -0.10    1     1
 5     -0.35       1.35      0.20    0     1
 6      0.35      -1.35     -0.20    1     0
 7      1.35       0.35      0.20    1     1
 8     -1.35      -0.35     -0.20    0     0


## Baseline（基线）：单一一维 shared-bottom

一个共享标量瓶颈必须把兴趣与价格压到同一轴，两个线性任务头只能在这条轴上设不同方向/阈值，无法同时表示二维平面中的“兴趣正负”和“价格正负”。我们先真实训练它，而不是直接假定负迁移。

In [2]:
class SharedScalarBaseline(nn.Module):  # 定义只有一个共享隐藏标量的硬共享基线。
    def __init__(self, input_dim=3, task_count=2):  # 初始化共享投影与两个任务头。
        super().__init__()  # 注册基础模块状态。
        self.shared_weight = nn.Parameter(torch.randn(input_dim, 1) * 0.25)  # 把三维输入压成单一共享轴。
        self.shared_bias = nn.Parameter(torch.zeros(1))  # 创建共享标量偏置。
        self.task_weight = nn.Parameter(torch.randn(1, task_count) * 0.20)  # 创建点击与转化两个标量头。
        self.task_bias = nn.Parameter(torch.zeros(task_count))  # 创建双任务偏置。
    def forward(self, features, return_hidden=False):  # 执行一维共享表示与双任务预测。
        hidden = torch.tanh(features @ self.shared_weight + self.shared_bias)  # 生成唯一共享瓶颈表示。
        logits = hidden @ self.task_weight + self.task_bias  # 从同一轴预测点击和转化。
        if return_hidden:  # 教学观察模式需要返回瓶颈值。
            return logits, hidden  # 返回双任务 logits 与共享标量。
        return logits  # 普通训练模式只返回 logits。
def train_multitask(model, epochs, learning_rate):  # 定义共享的双任务真实训练循环。
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)  # 创建更新手写参数的 Adam。
    history = []  # 保存总 loss、两个任务准确率和首参数梯度。
    for epoch in range(epochs):  # 重复优化全部训练曝光。
        optimizer.zero_grad()  # 清除上一轮累计梯度。
        logits = model(train_features)  # 调用当前模型的手写 forward。
        click_loss = F.binary_cross_entropy_with_logits(logits[:, 0], train_targets[:, 0])  # 计算点击任务二元交叉熵。
        conversion_loss = F.binary_cross_entropy_with_logits(logits[:, 1], train_targets[:, 1])  # 计算转化任务二元交叉熵。
        loss = click_loss + conversion_loss  # 以相同权重联合优化两个任务。
        loss.backward()  # 把两任务梯度共同传播到共享结构。
        first_parameter = next(model.parameters())  # 取得首个共享参数观察真实梯度。
        gradient_norm = float(first_parameter.grad.norm().detach())  # 计算共享参数梯度二范数。
        optimizer.step()  # 根据联合 loss 更新参数。
        predictions = (torch.sigmoid(logits) >= 0.5).float()  # 把训练概率转换成二元决策。
        accuracies = (predictions == train_targets).float().mean(dim=0)  # 分别计算点击和转化训练准确率。
        history.append((float(loss.detach()), float(accuracies[0]), float(accuracies[1]), gradient_norm))  # 保存可读训练轨迹。
    return history  # 返回优化过程供后续同口径比较。
torch.manual_seed(101)  # 固定 shared-bottom 参数初始化。
baseline_model = SharedScalarBaseline()  # 创建一维硬共享基线。
baseline_history = train_multitask(baseline_model, epochs=500, learning_rate=0.025)  # 真实训练基线到稳定状态。
with torch.no_grad():  # 关闭基线留出评估梯度。
    baseline_logits, baseline_hidden = baseline_model(test_features, return_hidden=True)  # 获取测试预测和共享标量。
    baseline_probabilities = torch.sigmoid(baseline_logits)  # 转换为点击与转化概率。
baseline_predictions = (baseline_probabilities >= 0.5).float()  # 形成双任务二元预测。
baseline_accuracies = (baseline_predictions == test_targets).float().mean(dim=0)  # 计算两个任务留出准确率。
baseline_average_accuracy = float(baseline_accuracies.mean())  # 计算双任务宏平均准确率。
print("阶段      loss    点击训练准确率  转化训练准确率  共享梯度")  # 输出基线训练过程表头。
for epoch in (0, 99, 499):  # 选择三个关键训练阶段。
    row = baseline_history[epoch]  # 读取当前阶段记录。
    print(f"{epoch + 1:>3}     {row[0]:>6.4f}       {row[1]:>6.1%}          {row[2]:>6.1%}       {row[3]:>7.4f}")  # 展示单瓶颈的优化上限。
print("编号  hidden   点击预测/gold  转化预测/gold")  # 输出逐测试曝光基线表头。
for index in range(len(test_records)):  # 遍历八条留出曝光。
    print(f"{index + 1:>2}    {float(baseline_hidden[index]):>6.3f}       {int(baseline_predictions[index, 0])}/{int(test_targets[index, 0])}           {int(baseline_predictions[index, 1])}/{int(test_targets[index, 1])}")  # 展示一维压缩导致的任务错误。
print(f"Shared-bottom 测试准确率：点击={float(baseline_accuracies[0]):.1%}，转化={float(baseline_accuracies[1]):.1%}，宏平均={baseline_average_accuracy:.1%}")  # 汇总双任务基线。

阶段      loss    点击训练准确率  转化训练准确率  共享梯度
  1     1.3813        12.5%           62.5%        0.1622
100     0.7831        50.0%          100.0%        0.0303
500     0.7010        50.0%          100.0%        0.0016
编号  hidden   点击预测/gold  转化预测/gold
 1     0.990       1/0           0/0
 2    -0.990       1/0           1/1
 3     0.990       1/1           0/0
 4    -0.990       1/1           1/1
 5    -0.999       1/0           1/1
 6     0.999       1/1           0/0
 7    -0.770       1/1           1/1
 8     0.770       1/0           0/0
Shared-bottom 测试准确率：点击=50.0%，转化=100.0%，宏平均=75.0%


## 公平容量对照：12 维 shared-bottom 与任务梯度夹角

一维基线是“信息瓶颈压力测试”，不能单独证明 MMoE 优于普通多任务网络。下面加入 12 维 shared-bottom：它有 74 个参数，接近本实验 MMoE 的 82 个参数。我们在相同数据和训练轮数下比较，同时在一个独立初始化快照上分别反传点击 loss 与转化 loss，计算共享权重梯度余弦；负值才是当前批次存在梯度冲突的直接证据。

In [3]:
class FairSharedBottom(nn.Module):  # 定义与 MMoE 参数量接近的向量共享基线。
    def __init__(self, input_dim=3, hidden_dim=12, task_count=2):  # 初始化十二维共享层与双任务头。
        super().__init__()  # 注册基础模块状态。
        self.shared_weight = nn.Parameter(torch.randn(input_dim, hidden_dim) * 0.25)  # 创建能同时保存兴趣与价格的共享表示。
        self.shared_bias = nn.Parameter(torch.zeros(hidden_dim))  # 创建共享隐藏偏置。
        self.task_weight = nn.Parameter(torch.randn(hidden_dim, task_count) * 0.20)  # 创建点击与转化两个任务头。
        self.task_bias = nn.Parameter(torch.zeros(task_count))  # 创建双任务偏置。
    def forward(self, features, return_hidden=False):  # 执行公平容量的 shared-bottom 前向。
        hidden = torch.tanh(features @ self.shared_weight + self.shared_bias)  # 生成十二维共享表示。
        logits = hidden @ self.task_weight + self.task_bias  # 从同一向量预测点击和转化。
        if return_hidden:  # 教学观察模式需要返回共享表示。
            return logits, hidden  # 返回双任务分数与共享向量。
        return logits  # 普通训练模式只返回双任务分数。
torch.manual_seed(102)  # 固定公平 shared-bottom 参数初始化。
fair_model = FairSharedBottom()  # 创建与 MMoE 参数量接近的共享模型。
fair_history = train_multitask(fair_model, epochs=500, learning_rate=0.025)  # 用完全相同训练协议优化公平基线。
with torch.no_grad():  # 关闭公平基线留出评估梯度。
    fair_logits, fair_hidden = fair_model(test_features, return_hidden=True)  # 获取八条测试曝光的双任务分数。
    fair_probabilities = torch.sigmoid(fair_logits)  # 把分数转换为点击与转化概率。
fair_predictions = (fair_probabilities >= 0.5).float()  # 形成双任务二元决策。
fair_accuracies = (fair_predictions == test_targets).float().mean(dim=0)  # 分别计算点击和转化准确率。
fair_average_accuracy = float(fair_accuracies.mean())  # 计算公平基线宏平均准确率。
fair_parameter_count = sum(parameter.numel() for parameter in fair_model.parameters())  # 统计公平基线全部参数。
mmoe_parameter_budget = 3 * (3 * 4 + 4) + 2 * (3 * 3 + 3) + 2 * (4 + 1)  # 按专家、gate 和 tower 公式计算 MMoE 参数预算。
torch.manual_seed(104)  # 固定梯度诊断模型初始化。
diagnostic_model = FairSharedBottom()  # 创建未训练快照避免收敛后梯度接近零。
diagnostic_logits = diagnostic_model(train_features)  # 对全部训练曝光执行一次前向。
diagnostic_click_loss = F.binary_cross_entropy_with_logits(diagnostic_logits[:, 0], train_targets[:, 0])  # 单独计算点击任务 loss。
diagnostic_conversion_loss = F.binary_cross_entropy_with_logits(diagnostic_logits[:, 1], train_targets[:, 1])  # 单独计算转化任务 loss。
click_gradient = torch.autograd.grad(diagnostic_click_loss, diagnostic_model.shared_weight, retain_graph=True)[0].reshape(-1)  # 提取点击 loss 对共享权重的梯度。
conversion_gradient = torch.autograd.grad(diagnostic_conversion_loss, diagnostic_model.shared_weight)[0].reshape(-1)  # 提取转化 loss 对同一共享权重的梯度。
gradient_cosine = float(F.cosine_similarity(click_gradient.unsqueeze(0), conversion_gradient.unsqueeze(0)).item())  # 计算两任务共享梯度夹角余弦。
print("公平对照       参数量  点击准确率  转化准确率  宏平均")  # 输出容量可比结果表头。
print(f"一维压力测试      {sum(parameter.numel() for parameter in baseline_model.parameters()):>3}      {float(baseline_accuracies[0]):>6.1%}      {float(baseline_accuracies[1]):>6.1%}   {baseline_average_accuracy:>6.1%}")  # 展示受限瓶颈结果。
print(f"12维Shared      {fair_parameter_count:>3}      {float(fair_accuracies[0]):>6.1%}      {float(fair_accuracies[1]):>6.1%}   {fair_average_accuracy:>6.1%}")  # 展示公平 shared-bottom 结果。
print(f"MMoE预算         {mmoe_parameter_budget:>3}      尚未训练      尚未训练       -")  # 在 MMoE 训练前展示参数预算。
print(f"独立初始化上的共享梯度余弦={gradient_cosine:.4f}；负值才表示该批次存在直接梯度冲突。")  # 给出可解释的负迁移诊断而非先验断言。

公平对照       参数量  点击准确率  转化准确率  宏平均
一维压力测试        8       50.0%      100.0%    75.0%
12维Shared       74      100.0%      100.0%   100.0%
MMoE预算          82      尚未训练      尚未训练       -
独立初始化上的共享梯度余弦=0.0496；负值才表示该批次存在直接梯度冲突。


## 核心实现：多专家与逐任务 gate

三个 expert 各自把输入映射成四维表示，堆成 `[B,3,4]`。每个任务的 gate 用自己的权重生成 `[B,3]` 概率，再通过 `gate.unsqueeze(-1) * expert_outputs` 沿 expert 维求和；最后点击与转化 tower 分别输出 logit。

In [4]:
class ManualMMoE(nn.Module):  # 定义不用现成推荐层的多门控专家模型。
    def __init__(self, input_dim=3, expert_count=3, expert_dim=4, task_count=2):  # 初始化专家、任务 gate 和任务 tower。
        super().__init__()  # 注册基础模块状态。
        self.expert_count = expert_count  # 保存共享专家数量。
        self.task_count = task_count  # 保存点击与转化任务数量。
        self.expert_weights = nn.ParameterList([nn.Parameter(torch.randn(input_dim, expert_dim) * 0.30) for _ in range(expert_count)])  # 创建三个独立专家投影。
        self.expert_biases = nn.ParameterList([nn.Parameter(torch.zeros(expert_dim)) for _ in range(expert_count)])  # 创建每个专家的偏置。
        self.gate_weights = nn.ParameterList([nn.Parameter(torch.randn(input_dim, expert_count) * 0.18) for _ in range(task_count)])  # 为每个任务创建独立 gate。
        self.gate_biases = nn.ParameterList([nn.Parameter(torch.zeros(expert_count)) for _ in range(task_count)])  # 创建逐任务 gate 偏置。
        self.tower_weights = nn.ParameterList([nn.Parameter(torch.randn(expert_dim, 1) * 0.22) for _ in range(task_count)])  # 创建逐任务输出 tower。
        self.tower_biases = nn.ParameterList([nn.Parameter(torch.zeros(1)) for _ in range(task_count)])  # 创建逐任务输出偏置。
    def forward(self, features, return_details=False):  # 执行专家计算、任务门控和双任务预测。
        expert_list = []  # 收集三个专家的 batch 表示。
        for weight, bias in zip(self.expert_weights, self.expert_biases):  # 遍历共享专家参数。
            expert_list.append(torch.tanh(features @ weight + bias))  # 计算当前专家的非线性表示。
        expert_outputs = torch.stack(expert_list, dim=1)  # 堆叠为 batch、expert、hidden 三维张量。
        task_logits = []  # 收集点击与转化输出。
        task_gates = []  # 收集逐任务专家概率以便诊断。
        task_representations = []  # 收集门控加权后的任务表示。
        for task_index in range(self.task_count):  # 分别计算两个任务的 gate 与 tower。
            gate_logits = features @ self.gate_weights[task_index] + self.gate_biases[task_index]  # 计算当前任务对三个专家的偏好分数。
            gate = torch.softmax(gate_logits, dim=-1)  # 沿专家维归一化且每条样本权重和为一。
            representation = (gate.unsqueeze(-1) * expert_outputs).sum(dim=1)  # 按 gate 汇聚共享专家输出。
            logit = (representation @ self.tower_weights[task_index] + self.tower_biases[task_index]).squeeze(-1)  # 用当前任务 tower 计算标量分数。
            task_logits.append(logit)  # 保存当前任务分数。
            task_gates.append(gate)  # 保存当前任务 gate。
            task_representations.append(representation)  # 保存当前任务专属表示。
        logits = torch.stack(task_logits, dim=-1)  # 合并为 batch、两个任务的输出。
        gates = torch.stack(task_gates, dim=1)  # 合并为 batch、task、expert 的门控张量。
        representations = torch.stack(task_representations, dim=1)  # 合并为 batch、task、hidden 的表示。
        if return_details:  # 教学观察模式返回专家、gate 与任务表示。
            return logits, expert_outputs, gates, representations  # 暴露 MMoE 全部关键中间量。
        return logits  # 普通训练模式只返回双任务 logits。
torch.manual_seed(103)  # 固定 MMoE 参数初始化。
mmoe = ManualMMoE()  # 创建三专家双门控模型。
preview_logits, preview_experts, preview_gates, preview_representations = mmoe(test_features[:2], return_details=True)  # 对两个留出曝光执行一次前向。
print("输入 / expert输出 / gate / 任务表示 / logits：", tuple(test_features[:2].shape), tuple(preview_experts.shape), tuple(preview_gates.shape), tuple(preview_representations.shape), tuple(preview_logits.shape))  # 展示公式对应的真实张量形状。
print("样本1点击 gate：", [round(float(value), 3) for value in preview_gates[0, 0]])  # 展示随机初始化的点击专家权重。
print("样本1转化 gate：", [round(float(value), 3) for value in preview_gates[0, 1]])  # 展示同一样本不同任务拥有独立门控。
print("两任务 gate 行和：", [round(float(value), 6) for value in preview_gates[0].sum(dim=-1)])  # 观察 softmax 合同而不用断言替代教学。

输入 / expert输出 / gate / 任务表示 / logits： (2, 3) (2, 3, 4) (2, 2, 3) (2, 2, 4) (2, 2)
样本1点击 gate： [0.271, 0.367, 0.362]
样本1转化 gate： [0.235, 0.259, 0.506]
两任务 gate 行和： [1.0, 1.0]


## 真实联合训练与逐样本结果

MMoE 使用与基线完全相同的训练样本、两个 BCE loss 和宏平均准确率。除了 loss 与梯度，还输出每条测试曝光的双任务概率，以及点击/转化 gate 的专家分配，观察任务是否形成不同路由。

In [5]:
mmoe_history = train_multitask(mmoe, epochs=500, learning_rate=0.025)  # 在相同三十二条曝光上真实训练 MMoE。
with torch.no_grad():  # 关闭留出评估梯度。
    mmoe_logits, trained_experts, trained_gates, trained_representations = mmoe(test_features, return_details=True)  # 获取双任务预测和门控证据。
    mmoe_probabilities = torch.sigmoid(mmoe_logits)  # 把 logits 转换为点击和转化概率。
mmoe_predictions = (mmoe_probabilities >= 0.5).float()  # 形成双任务二元决策。
mmoe_accuracies = (mmoe_predictions == test_targets).float().mean(dim=0)  # 分别计算点击和转化测试准确率。
mmoe_average_accuracy = float(mmoe_accuracies.mean())  # 计算双任务宏平均准确率。
mmoe_parameter_count = sum(parameter.numel() for parameter in mmoe.parameters())  # 统计 MMoE 全部专家、gate 与 tower 参数。
print("阶段      loss    点击训练准确率  转化训练准确率  首专家梯度")  # 输出 MMoE 训练过程表头。
for epoch in (0, 99, 499):  # 选择三个关键训练阶段。
    row = mmoe_history[epoch]  # 读取当前阶段训练记录。
    print(f"{epoch + 1:>3}     {row[0]:>6.4f}       {row[1]:>6.1%}          {row[2]:>6.1%}       {row[3]:>7.4f}")  # 展示专家参数被真实更新。
print("编号  点击概率/gold  转化概率/gold  点击gate          转化gate")  # 输出逐曝光结果和路由表头。
for index in range(len(test_records)):  # 遍历八条留出曝光。
    click_gate = [round(float(value), 2) for value in trained_gates[index, 0]]  # 格式化点击任务专家权重。
    conversion_gate = [round(float(value), 2) for value in trained_gates[index, 1]]  # 格式化转化任务专家权重。
    print(f"{index + 1:>2}      {float(mmoe_probabilities[index, 0]):.3f}/{int(test_targets[index, 0])}          {float(mmoe_probabilities[index, 1]):.3f}/{int(test_targets[index, 1])}       {click_gate}    {conversion_gate}")  # 展示概率、标签和任务路由。
print(f"测试准确率：标量压力测试={baseline_average_accuracy:.1%}，公平Shared={fair_average_accuracy:.1%}，MMoE={mmoe_average_accuracy:.1%}；参数量 Fair={fair_parameter_count} / MMoE={mmoe_parameter_count}")  # 公平汇总容量与同数据指标且不夸大 MMoE 优势。
average_gate_by_task = trained_gates.mean(dim=0)  # 计算测试集上每个任务的平均专家负载。
print("点击任务平均 gate：", [round(float(value), 3) for value in average_gate_by_task[0]])  # 展示点击路由偏好。
print("转化任务平均 gate：", [round(float(value), 3) for value in average_gate_by_task[1]])  # 展示转化路由偏好。

阶段      loss    点击训练准确率  转化训练准确率  首专家梯度
  1     1.3366        81.2%           81.2%        0.0525
100     0.0151       100.0%          100.0%        0.0011
500     0.0013       100.0%          100.0%        0.0001
编号  点击概率/gold  转化概率/gold  点击gate          转化gate
 1      0.000/0          0.000/0       [0.01, 0.94, 0.04]    [0.87, 0.02, 0.12]
 2      0.000/0          1.000/1       [0.01, 0.03, 0.96]    [0.01, 0.99, 0.0]
 3      1.000/1          0.000/0       [0.99, 0.01, 0.0]    [0.04, 0.02, 0.94]
 4      1.000/1          1.000/1       [0.99, 0.0, 0.01]    [0.0, 0.98, 0.02]
 5      0.001/0          1.000/1       [0.06, 0.01, 0.93]    [0.0, 1.0, 0.0]
 6      0.968/1          0.001/0       [0.72, 0.28, 0.0]    [0.18, 0.0, 0.81]
 7      1.000/1          0.985/1       [1.0, 0.0, 0.0]    [0.0, 0.75, 0.24]
 8      0.001/0          0.003/0       [0.0, 0.69, 0.31]    [0.86, 0.11, 0.04]
测试准确率：标量压力测试=75.0%，公平Shared=100.0%，MMoE=100.0%；参数量 Fair=74 / MMoE=82
点击任务平均 gate： [0.472, 0.245, 0.282]
转化任务平均 

## 结果解读

一维 shared-bottom 的 75% 只证明标量瓶颈会丢信息；参数量接近的 12 维 shared-bottom 也能解出这组干净的正交任务，因此不能把 MMoE 的 100% 宣称为结构收益。MMoE 在这里真正展示的是逐任务 gate 和专家分工机制。梯度余弦提供冲突诊断，但一次小批结果不代表线上负迁移；要证明收益还需多随机种子、同参数/同 FLOPs 对照和真实噪声任务。gate 权重是路由诊断，不等于因果解释。

## 失败案例：gate softmax 错沿 batch 维归一化

gate 应让“每条曝光的三个专家权重和为 1”。若写成 `dim=0`，权重会在不同曝光之间竞争，同一曝光单独预测和放进批次预测会改变表示，在线 batch size 变化就产生漂移。下面直接用训练后点击 gate 复现。

In [6]:
probe_sample = test_features[:1]  # 选择第一条曝光作为批不变性探针。
comparison_batch = test_features[:4]  # 构造包含同一曝光的四样本批次。
def expert_stack(model, features):  # 用训练后参数计算全部专家表示。
    outputs = []  # 收集逐专家输出。
    for weight, bias in zip(model.expert_weights, model.expert_biases):  # 遍历三个训练后专家。
        outputs.append(torch.tanh(features @ weight + bias))  # 计算当前专家表示。
    return torch.stack(outputs, dim=1)  # 返回 batch、expert、hidden 张量。
single_experts = expert_stack(mmoe, probe_sample)  # 计算单样本的专家输出。
batch_experts = expert_stack(mmoe, comparison_batch)  # 计算四样本批次专家输出。
single_gate_logits = probe_sample @ mmoe.gate_weights[0] + mmoe.gate_biases[0]  # 计算单样本点击 gate logits。
batch_gate_logits = comparison_batch @ mmoe.gate_weights[0] + mmoe.gate_biases[0]  # 计算批次点击 gate logits。
wrong_single_gate = torch.softmax(single_gate_logits, dim=0)  # 错误沿 batch 维对单样本归一化使每专家权重都为一。
wrong_batch_gate = torch.softmax(batch_gate_logits, dim=0)  # 错误让不同曝光竞争每个专家的概率质量。
correct_single_gate = torch.softmax(single_gate_logits, dim=-1)  # 正确沿专家维归一化单样本权重。
correct_batch_gate = torch.softmax(batch_gate_logits, dim=-1)  # 正确沿专家维独立归一化每条曝光。
wrong_single_representation = (wrong_single_gate.unsqueeze(-1) * single_experts).sum(dim=1)  # 计算错误单样本任务表示。
wrong_batch_representation = (wrong_batch_gate.unsqueeze(-1) * batch_experts).sum(dim=1)[0:1]  # 取同一曝光在批次中的错误表示。
correct_single_representation = (correct_single_gate.unsqueeze(-1) * single_experts).sum(dim=1)  # 计算正确单样本表示。
correct_batch_representation = (correct_batch_gate.unsqueeze(-1) * batch_experts).sum(dim=1)[0:1]  # 取同一曝光在批次中的正确表示。
wrong_drift = float((wrong_single_representation - wrong_batch_representation).abs().max())  # 量化错误 softmax 的批组成漂移。
correct_drift = float((correct_single_representation - correct_batch_representation).abs().max())  # 量化正确 softmax 的批不变性。
print("方案            单样本gate和  批内同样本gate和  表示最大漂移")  # 输出 gate 轴错误对比表头。
print(f"错误dim=0          {float(wrong_single_gate.sum()):.3f}          {float(wrong_batch_gate[0].sum()):.3f}           {wrong_drift:.6f}")  # 展示权重合同和表示同时失效。
print(f"正确dim=-1         {float(correct_single_gate.sum()):.3f}          {float(correct_batch_gate[0].sum()):.3f}           {correct_drift:.6f}")  # 展示逐样本路由不受 batch 组成影响。
print("错误单样本 gate / 批内 gate：", [round(float(value), 3) for value in wrong_single_gate[0]], [round(float(value), 3) for value in wrong_batch_gate[0]])  # 展示同一请求的实际路由变化。

方案            单样本gate和  批内同样本gate和  表示最大漂移
错误dim=0          3.000          0.916           2.065109
正确dim=-1         1.000          1.000           0.000000
错误单样本 gate / 批内 gate： [1.0, 1.0, 1.0] [0.003, 0.897, 0.016]


## 生产差距与追问

线上点击与转化标签存在延迟和缺失，需逐任务 label mask；转化只在点击后可见会产生样本选择偏差，可用 ESMM 等漏斗建模。还要按用户时间切分、处理任务量级与 loss 权重、监控 gate 熵/专家负载/梯度夹角、校准各任务概率，并在召回、排序延迟与业务约束下做 A/B 测试。更多 expert 不一定更好，计算成本和塌缩都需实测。

## 最小回归测试

In [7]:
assert len(test_records) >= 5  # 保证多任务案例包含足够多任务冲突组合。
assert fair_average_accuracy >= 0.9  # 保护公平容量 shared-bottom 能揭示标量基线的容量混杂。
assert mmoe_average_accuracy >= 0.9  # 保护手写 MMoE 在两个任务上都形成有效预测。
assert float(mmoe_accuracies.min()) >= 0.9  # 保护点击和转化两个任务都获得有效预测。
assert correct_drift < 1e-7  # 保护正确 gate 归一化保持批组成不变性。
assert wrong_drift > 1e-3  # 保护失败案例确实复现错误 softmax 轴漂移。
print("最小回归测试通过：双任务训练、专家路由和 gate 轴故障修复均保持有效。")  # 输出集中测试结论。

最小回归测试通过：双任务训练、专家路由和 gate 轴故障修复均保持有效。
